# ⚡ Module 5: Support Vector Machines (SVM)
## Theory + Practical — Complete Guide

---

## 📚 THEORY SECTION

### What is SVM?

**Support Vector Machine** finds the **optimal hyperplane** that maximally separates classes in feature space.

### Key Concepts

#### 1. Hyperplane
In N-dimensional space, a hyperplane has N-1 dimensions:
```
2D: a line         (w₁x₁ + w₂x₂ + b = 0)
3D: a plane        (w₁x₁ + w₂x₂ + w₃x₃ + b = 0)
ND: hyperplane     (wᵀx + b = 0)
```

#### 2. Margin
The **margin** is the distance between the hyperplane and the nearest training points (support vectors).

```
Margin = 2 / ‖w‖

SVM Objective: Maximize margin (minimize ‖w‖²)
```

#### 3. Support Vectors
The **support vectors** are the training points closest to the decision boundary. They define the margin and are the only points that matter for the hyperplane!

### Hard Margin vs Soft Margin

| Type | When | What |
|------|------|------|
| **Hard Margin** | Linearly separable data | No misclassifications allowed |
| **Soft Margin** | Non-linearly separable | Allows some misclassifications (controlled by C) |

#### C Parameter (Regularization)
```
C small  → Wide margin, more misclassifications (more regularization)
C large  → Narrow margin, fewer misclassifications (less regularization)
```

### The Kernel Trick

**Problem**: What if data is not linearly separable in original space?

**Solution**: Project data to higher-dimensional space where it IS separable!

```
Original space (2D) → Feature space (higher D) → Linear separation
```

**Common Kernels:**

| Kernel | Formula | Use Case |
|--------|---------|----------|
| **Linear** | K(x,z) = xᵀz | Linearly separable, high-dim data |
| **RBF/Gaussian** | K(x,z) = exp(-γ‖x-z‖²) | Most common, works well generally |
| **Polynomial** | K(x,z) = (xᵀz + c)ᵈ | Polynomial relationships |
| **Sigmoid** | K(x,z) = tanh(αxᵀz + c) | Neural network-like |

### RBF Kernel Parameter γ (Gamma)
```
γ small  → Wide influence, smoother boundary (underfitting risk)
γ large  → Narrow influence, complex boundary (overfitting risk)
```

---

## 🔬 PRACTICAL SECTION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC, SVR
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('✅ Libraries loaded!')

### 🔬 Practical 1: Understanding Margins & Support Vectors

In [ ]:
# ============================================================
# PRACTICAL 1: Visualizing SVM margin and support vectors
# ============================================================

def plot_svm_decision_boundary(X, y, model, ax, title='SVM Decision Boundary'):
    """Plot SVM decision boundary, margins, and support vectors."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=1)
    
    # Plot decision function for margin visualization
    if hasattr(model, 'decision_function'):
        Z_df = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z_df = Z_df.reshape(xx.shape)
        ax.contour(xx, yy, Z_df, levels=[-1, 0, 1], 
                   linestyles=['--', '-', '--'], colors=['red', 'black', 'blue'],
                   linewidths=[1.5, 2, 1.5])
    
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', 
                          edgecolors='k', s=60, linewidth=0.5, zorder=5)
    
    # Highlight support vectors
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                   s=150, linewidth=2, facecolors='none', edgecolors='yellow',
                   zorder=10, label=f'Support Vectors ({len(model.support_vectors_)})')
        ax.legend(fontsize=8)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Generate linearly separable data
X_lin, y_lin = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42, class_sep=2.0)
scaler = StandardScaler()
X_lin_s = scaler.fit_transform(X_lin)

# Train SVMs with different C values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SVM: Effect of C Parameter on Margin', fontsize=14, fontweight='bold')

for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    svm = SVC(kernel='linear', C=C, random_state=42)
    svm.fit(X_lin_s, y_lin)
    acc = accuracy_score(y_lin, svm.predict(X_lin_s))
    plot_svm_decision_boundary(X_lin_s, y_lin, svm, ax,
                                f'C={C}\n(Acc={acc:.2f}, SVs={svm.n_support_.sum()})')

plt.tight_layout()
plt.show()

print('📌 Observations:')
print('  C=0.01: Wide margin, more support vectors, some misclassifications')
print('  C=1.0:  Balanced margin')
print('  C=100:  Narrow margin, fewer support vectors, tries to classify all correctly')

### 🔬 Practical 2: Kernel Trick — Making Non-Linear Data Separable

In [ ]:
# ============================================================
# PRACTICAL 2: Different kernels on different data shapes
# ============================================================

# Generate different types of non-linear data
X_circles, y_circles = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
X_linear, y_linear = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                          n_clusters_per_class=1, random_state=42, class_sep=1.5)

datasets = [
    (X_circles, y_circles, 'Circles Data'),
    (X_moons, y_moons, 'Moons Data'),
    (X_linear, y_linear, 'Linear Data')
]

kernels = ['linear', 'rbf', 'poly']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('SVM Kernels on Different Data Shapes', fontsize=14, fontweight='bold')

for row_idx, (X_d, y_d, data_name) in enumerate(datasets):
    scaler = StandardScaler()
    X_d_s = scaler.fit_transform(X_d)
    
    for col_idx, kernel in enumerate(kernels):
        ax = axes[row_idx, col_idx]
        svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
        svm.fit(X_d_s, y_d)
        acc = accuracy_score(y_d, svm.predict(X_d_s))
        
        # Decision boundary
        h = 0.03
        x_min, x_max = X_d_s[:, 0].min() - 0.5, X_d_s[:, 0].max() + 0.5
        y_min, y_max = X_d_s[:, 1].min() - 0.5, X_d_s[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
        ax.scatter(X_d_s[:, 0], X_d_s[:, 1], c=y_d, cmap='RdYlBu', 
                   edgecolors='k', s=20, linewidth=0.3)
        
        title = f'{data_name}\nKernel: {kernel.upper()} | Acc: {acc:.2f}'
        if acc < 0.75:
            title += ' ❌'
        elif acc > 0.95:
            title += ' ✅'
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

print('\n📌 Key Insight: RBF kernel works well on circles and moons (non-linear)!')
print('   Linear kernel only works on linearly separable data.')

### 🔬 Practical 3: SVM on Real Dataset + Hyperparameter Tuning

In [ ]:
# ============================================================
# PRACTICAL 3: SVM on Breast Cancer + C and gamma tuning
# ============================================================

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Grid search over C and gamma
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}

print('Running GridSearchCV for SVM...')
grid = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42),
                    param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_s, y_train)

# Heatmap of results
results = grid.cv_results_['mean_test_score'].reshape(4, 4)
C_vals = [0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(results, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[str(g) for g in gamma_vals],
            yticklabels=[str(c) for c in C_vals],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('GridSearchCV — Mean CV Accuracy (RBF Kernel)', fontweight='bold')
axes[0].set_xlabel('Gamma')
axes[0].set_ylabel('C')

# Final evaluation
best_svm = grid.best_estimator_
y_pred = best_svm.predict(X_test_s)
test_acc = accuracy_score(y_test, y_pred)

# Compare different SVMs
svm_configs = {
    'Linear SVM': SVC(kernel='linear', C=1.0),
    'RBF SVM (default)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'RBF SVM (tuned)': best_svm,
    'Poly SVM (d=3)': SVC(kernel='poly', degree=3, C=1.0)
}

names, scores = [], []
for name, svm in svm_configs.items():
    svm.fit(X_train_s, y_train)
    score = accuracy_score(y_test, svm.predict(X_test_s))
    names.append(name)
    scores.append(score)
    print(f'{name:25s}: {score:.4f}')

axes[1].barh(names, scores, color=['#3498DB', '#E74C3C', '#2ECC71', '#F39C12'],
             edgecolor='black', alpha=0.85)
axes[1].set_xlim(0.9, 1.01)
axes[1].set_title('SVM Variants — Test Accuracy', fontweight='bold')
axes[1].set_xlabel('Accuracy')
for i, score in enumerate(scores):
    axes[1].text(score + 0.001, i, f'{score:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n✅ Best Parameters: {grid.best_params_}')
print(f'✅ Best Test Accuracy: {test_acc:.4f}')

## 🎓 Summary

| Concept | Key Takeaway |
|---------|-------------|
| SVM Goal | Maximize margin between classes |
| Support Vectors | Only boundary points matter |
| C Parameter | Small C = wide margin (regularization); Large C = narrow margin |
| Kernel Trick | Project to higher space without computing it explicitly |
| RBF Kernel | Best for non-linear data; controlled by γ |
| γ Parameter | Small γ = smooth boundary; Large γ = complex boundary |
| Scaling | SVM is sensitive to feature scale — always scale! |

## 📖 Next: K-Nearest Neighbors (KNN)